# Sales & Customer Behavior Analysis

A basic portfolio project using synthetic retail transaction data to explore sales performance and customer purchasing behavior.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv('../data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('../data/customers.csv')
products = pd.read_csv('../data/products.csv')

orders.head()

## 2. Data Understanding

In [ ]:
print('Orders:', orders.shape)
print('Customers:', customers.shape)
print('Products:', products.shape)
print('\nMissing values:')
print(orders.isna().sum())
print('\nOrder status:')
print(orders['order_status'].value_counts())

## 3. Prepare Analysis Dataset

In [ ]:
df = orders.merge(customers, on='customer_id').merge(
    products[['product_id', 'product', 'category']], on='product_id'
)
df['revenue'] = df['quantity'] * df['unit_price']
completed = df[df['order_status'] == 'Completed'].copy()
completed.head()

## 4. Key Metrics

In [ ]:
total_revenue = completed['revenue'].sum()
completed_orders = completed['order_id'].nunique()
aov = total_revenue / completed_orders
repeat_customer_rate = completed.groupby('customer_id')['order_id'].nunique().gt(1).mean() * 100

print(f'Total revenue: {total_revenue:,.2f}')
print(f'Completed orders: {completed_orders:,}')
print(f'Average order value: {aov:,.2f}')
print(f'Repeat customer rate: {repeat_customer_rate:.2f}%')

## 5. Revenue by Category

In [ ]:
category_summary = completed.groupby('category').agg(
    orders=('order_id','nunique'),
    units=('quantity','sum'),
    revenue=('revenue','sum')
).sort_values('revenue', ascending=False)

category_summary

In [ ]:
category_summary['revenue'].plot(
    kind='bar', figsize=(8,4.5), title='Revenue by Category'
)
plt.ylabel('Revenue')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 6. Top Products

In [ ]:
product_summary = completed.groupby(['product','category']).agg(
    orders=('order_id','nunique'),
    units=('quantity','sum'),
    revenue=('revenue','sum')
).sort_values('revenue', ascending=False)

product_summary.head(10)

## 7. Monthly Revenue

In [ ]:
monthly = (
    completed.assign(month=completed['order_date'].dt.to_period('M').astype(str))
    .groupby('month')
    .agg(orders=('order_id','nunique'), revenue=('revenue','sum'))
    .reset_index()
)
monthly

In [ ]:
plt.figure(figsize=(8,4.5))
plt.plot(monthly['month'], monthly['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Customer Behavior

In [ ]:
customer_summary = completed.groupby('customer_id').agg(
    orders=('order_id','nunique'),
    revenue=('revenue','sum')
).sort_values('revenue', ascending=False)

customer_summary.head(10)

## 9. Key Insights

Write observations from the results rather than assumptions. Suggested questions:

- Which category contributes the most revenue?
- Which products are the largest revenue contributors?
- Are there noticeable monthly trends?
- What is the repeat customer rate?
- Which cities generate the most revenue?
- What share of orders are cancelled or pending?

**Note:** The dataset is synthetic and is used for learning and portfolio demonstration.